# Project B (SumQA): starter notebook

HY562 Fall 2026. This notebook runs the whole pipeline on the toy KG (no server, no LLM needed),
shows where your summary code goes, and how to run the real benchmarks.
Run it from the `kit` folder of Project B inside the course container.

In [ ]:
import json, os, subprocess, sys
sys.path.insert(0, os.getcwd())
os.environ.setdefault("TOY_ENDPOINT", "file:data/toy/kg.nt")
# Real KGs (after SETUP.md, steps 4 to 6):
# os.environ["DBPEDIA_ENDPOINT"] = "oxigraph:data/kg/dbpedia2016/store"
# os.environ["DBLP_ENDPOINT"] = "oxigraph:data/kg/dblp2022/store"
# os.environ["OLLAMA_HOST"] = "http://host.docker.internal:11434"   # Ollama running on your laptop

def sh(cmd):
    """Run a kit command and print its output (the same commands work in a terminal)."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:] if r.returncode else "")

TEAM = "B1"   # your team

## 1. The toy pipeline, end to end
`mock:gold` returns the gold query and `mock:noisy` breaks every second query, so you can check the scorer.

In [ ]:
sh("python scripts/make_toy.py")
sh("python -m sumqa.gold --bench data/toy/toy_test.jsonl --endpoint $TOY_ENDPOINT")
sh(f"python -m sumqa.run --bench data/toy/toy_test.gold.jsonl --kg toy --context retrieval --budget 150 --model mock:noisy --team {TEAM}")
sh(f"python -m sumqa.score runs/{TEAM}/toy_test__retrieval__150__mock-noisy.jsonl")

## 2. Look at a run
Every record keeps the question, the generated query, its answers, the gold answers and the token counts.

In [ ]:
rows = [json.loads(l) for l in open(f"runs/{TEAM}/toy_test__retrieval__150__mock-noisy.jsonl")]
r = rows[1]
print(r["question"]); print(r["pred_sparql"]); print(r["answers"], "gold:", r["gold_answers"])
print("context tokens:", r["context_tokens"], "lines:", r["context_lines"])

## 3. Your summary (PySpark)

Load the KG as a DataFrame with columns `s, p, o, okind, dt, lang`. For DBpedia and dblp, `sumqa.kgstats --parquet` writes Parquet
files (SETUP.md, step 4); for the toy KG we parse the N-Triples file directly.

Write `compute_summary(triples) -> dict` in the summary format of `sumqa/context.py`
(nodes, edges, optional attributes). The placeholder below puts all entities into one node, so that the rest of the notebook runs.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from sumqa.kgstats import load_triples, RDF_TYPE

spark = SparkSession.builder.appName("projectB").getOrCreate()
triples = load_triples(spark, ["data/toy/kg.nt"])          # real KG: spark.read.parquet("data/kg/dbpedia2016/parquet")
triples.groupBy("okind").count().show()

def compute_summary(t):
    """TODO (your team's variant): weak / strong quotient summary (B1), type-based and pattern
    summary with frequencies (B2), or the strong summary that your slicer works on (B3)."""
    edges = (t.filter((F.col("okind") == "iri") & (F.col("p") != RDF_TYPE))
               .groupBy("p").count().collect())
    n = t.select("s").distinct().count()
    return {"kg": "toy", "kind": "placeholder", "team": TEAM,
            "nodes": [{"id": "n1", "classes": [], "size": n, "properties": []}],
            "edges": [{"source": "n1", "property": r["p"], "target": "n1", "count": r["count"]} for r in edges]}

summary = compute_summary(triples)
os.makedirs(f"runs/{TEAM}", exist_ok=True)
json.dump(summary, open(f"runs/{TEAM}/summary_toy.json", "w"), indent=1)

## 4. What the model will see
The serializer orders edges by count and puts each node description before its first edge; the budget cuts the list.
Check how much of your summary fits in 500, 1000 and 2000 tokens.

In [ ]:
from sumqa.context import summary_lines, cut_to_budget
from sumqa.tokens import count_tokens, COUNTER
lines = summary_lines(summary)
print(len(lines), "lines,", count_tokens("\n".join(lines)), "tokens", f"({COUNTER})")
for b in (500, 1000, 2000):
    print(b, "->", len(cut_to_budget(lines, b)), "lines")
print("\n".join(lines[:10]))

In [ ]:
sh(f"python -m sumqa.run --bench data/toy/toy_test.gold.jsonl --kg toy --context summary --summary runs/{TEAM}/summary_toy.json --variant mysummary --budget 300 --model mock:gold --team {TEAM}")
sh(f"python -m sumqa.score runs/{TEAM}/toy_test__mysummary__300__mock-gold.jsonl --results results/results_B_toy.csv")

## 5. Question-specific contexts (Team B3)
Write one line list per question and use `--context lines`. The example gives every question the first 5 lines.

In [ ]:
qs = [json.loads(l) for l in open("data/toy/toy_test.gold.jsonl")]
with open(f"runs/{TEAM}/slices_toy.jsonl", "w") as f:
    for q in qs:
        f.write(json.dumps({"id": q["id"], "lines": lines[:5]}) + "\n")    # TODO: your slice for q
sh(f"python -m sumqa.run --bench data/toy/toy_test.gold.jsonl --kg toy --context lines --lines runs/{TEAM}/slices_toy.jsonl --variant slice --budget 300 --model mock:gold --team {TEAM}")

## 6. Real runs (after SETUP.md)

```bash
# once per model, on your laptop (outside the container): ollama pull qwen2.5-coder:7b
python -m sumqa.run --bench data/benchmarks/qald9plus_test.gold.jsonl --kg dbpedia2016 \
    --context ontology --budget 1000 --model ollama:qwen2.5-coder:7b --team B1
python -m sumqa.score runs/B1/qald9plus_test__ontology__1000__ollama-qwen2.5-coder-7b.jsonl \
    --results results/results_B.csv --errors runs/B1/errors_ontology_1000.csv
python -m sumqa.ablation --run runs/B1/qald9plus_test__strong__1000__ollama-qwen2.5-coder-7b.jsonl \
    --bench data/benchmarks/qald9plus_test.gold.jsonl --kg dbpedia2016 --summary runs/B1/strong_dbpedia.json
```
Start with `--limit 10` to check that everything works before a full run.